In [2]:
"""
PIPELINE STAGE: JSON-Array Metadata Extraction & Relational Transformation
INPUT: raw_data/geo_metadata/geo_lat_lon.json (Array-based Object Structure)
OUTPUT: processed_data/steps/10_a_geo_metadata_cleaned.xlsx
LIBRARIES: pandas, json, os

1. RESEARCH CORE: POLYGLOT DATA MINING
    The objective is to extract geospatial intelligence from an array-based JSON 
    structure. This stage demonstrates the pipeline's ability to handle polyglot 
    data formats, moving from unstructured text (Word) to semi-structured 
    sequential streams (CSV) and finally to structured object arrays (JSON).

2. EXTRACTION ARCHITECTURE:
    - Type-Aware Parsing: The algorithm implements an 'Instance Check' to detect 
      whether the JSON source is an associative dictionary or a sequential list, 
      ensuring robust data ingestion.
    - Token Mapping: Targeted extraction of [Plate], [Latitude], and [Longitude] 
      tokens. The script uses a multi-key lookup strategy (e.g., 'plate' vs 'id') 
      to maintain high recall despite potential metadata naming inconsistencies.

3. DATA NORMALIZATION & VALIDATION:
    - Structural Flattening: The nested object array is flattened into a 
      long-format relational table.
    - Spatial Audit: Post-extraction validation to confirm the presence of 
      all 81 administrative units and the precision of floating-point 
      geographic coordinates.

4. METHODOLOGICAL SIGNIFICANCE:
    Isolating the geospatial metadata into a dedicated Excel asset allows for 
    a secondary audit of spatial accuracy before it is merged into the 
    longitudinal 2020-2024 energy database for regional clustering analysis.
"""

import pandas as pd
import json
import os

def json_kordinat_ayristirma_liste():
    # Dosya Yolları
    JSON_PATH = os.path.join("..", "..", "raw_data", "geo_metadata", "geo_lat_lon.json")
    OUT_PATH = os.path.join("..", "..", "processed_data", "steps", "10_a_geo_metadata_cleaned.xlsx")
    
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    
    try:
        print(f"JSON dosyası okunuyor: {JSON_PATH}")
        with open(JSON_PATH, 'r', encoding='utf-8') as f:
            geo_data = json.load(f)
        
        geo_list = []
        
        # HATA ÇÖZÜMÜ: Eğer veri bir liste ise direkt üzerinde dönüyoruz
        if isinstance(geo_data, list):
            for item in geo_data:
                # Plaka bilgisini 'plate' veya 'id' anahtarından almaya çalışalım
                plate = item.get('plate') or item.get('id') or item.get('plaka')
                lat = item.get('lat') or item.get('latitude')
                lon = item.get('lon') or item.get('longitude')
                
                if plate is not None:
                    geo_list.append({
                        'Plate': int(plate),
                        'Latitude': lat,
                        'Longitude': lon
                    })
        
        # Eğer veri hala sözlük ise (tedbir amaçlı)
        elif isinstance(geo_data, dict):
            for plate, coords in geo_data.items():
                lat = coords.get('lat') or coords.get('latitude')
                lon = coords.get('lon') or coords.get('longitude')
                geo_list.append({
                    'Plate': int(plate),
                    'Latitude': lat,
                    'Longitude': lon
                })

        # DataFrame oluştur
        df_geo = pd.DataFrame(geo_list)
        
        if df_geo.empty:
            print("UYARI: JSON içinden veri çekilemedi. Anahtar isimlerini (lat, lon, plate) kontrol edin.")
            return

        # Plaka numarasına göre sırala
        df_geo = df_geo.sort_values(by='Plate', ascending=True)
        
        # Excel'e kaydet
        df_geo.to_excel(OUT_PATH, index=False)
        
        print("\n" + "="*40)
        print("BAŞARILI: Liste formatındaki JSON Excel'e dönüştürüldü.")
        print(f"Toplam Kayıt: {len(df_geo)}")
        print(f"Kayıt Yeri: {OUT_PATH}")
        print("="*40)

    except Exception as e:
        print(f"HATA: {e}")

if __name__ == "__main__":
    json_kordinat_ayristirma_liste()

JSON dosyası okunuyor: ..\..\raw_data\geo_metadata\geo_lat_lon.json

BAŞARILI: Liste formatındaki JSON Excel'e dönüştürüldü.
Toplam Kayıt: 81
Kayıt Yeri: ..\..\processed_data\steps\10_a_geo_metadata_cleaned.xlsx
